In [22]:
from mdcrow import MDCrow

In [23]:
agent = MDCrow(model="gpt-4o-2024-05-13")

In [24]:
prompt_template = """
DO NOT RUN ANYTHING — just answer.

If you were given the following prompt, choose the most appropriate
ensemble, thermostat, barostat, and force field. All will be simulated in OpenMM, so
choices should be compatible with OpenMM.

IMPORTANT:
- Justify why your chosen ensemble is appropriate
- Explicitly state why alternative ensembles are NOT required
- Justify your choice of thermostat and barostat (if applicable) 
- Justify your choice of force field
- List any assumptions you are making about the system, and how those assumptions impact your choices.
- Avoid overly general statements; be specific to the system described in the prompt.

Respond with bullet points for each choice. Be concise, but specific.

Example prompt:
{prompt}
"""


In [29]:
prompt0_equil = """Simulate (in OpenMM) the thermophilic serine proteinase
(PDB: 1DBI), which is experimentally stable at 75°C,
for 1 ns to assess whether its folded structure remains stable.
Assume we are starting with a system **already equilibrated** at 75°C and 1 bar.
"""

prompt0_equil = prompt_template.format(prompt=prompt0_equil)
response0_equil = agent.run(prompt0_equil)
print("Response 0_equil:", response0_equil[0]["output"])

- **Ensemble: NVT (Canonical Ensemble)**
  - **Justification**: Since the system is already equilibrated at 75°C and 1 bar, maintaining a constant number of particles (N), volume (V), and temperature (T) is appropriate. This allows us to focus on the stability of the folded structure without the need to adjust pressure.
  - **Alternative Ensembles**: 
    - **NPT (Isothermal-Isobaric Ensemble)**: Not required because the system is already equilibrated at the desired pressure.
    - **NVE (Microcanonical Ensemble)**: Not suitable as it does not control temperature, which is crucial for assessing stability at 75°C.

- **Thermostat: Langevin Thermostat**
  - **Justification**: The Langevin thermostat is effective at maintaining a constant temperature and is compatible with OpenMM. It also helps in damping out any fluctuations that might arise, ensuring the system remains at 75°C.
  - **Alternative Thermostats**: 
    - **Berendsen Thermostat**: Less accurate in maintaining temperature ove

In [30]:
prompt0_solv = """Simulate (in OpenMM) the thermophilic serine proteinase
(PDB: 1DBI), which is experimentally stable at 75°C,
for 1 ns to assess whether its folded structure remains stable.
Assume we are starting from a freshly solvated system.
"""

prompt0_solv = prompt_template.format(prompt=prompt0_solv)
response0_solv = agent.run(prompt0_solv)
print("Response 0_solv:", response0_solv[0]["output"])

- **Ensemble: NPT (isothermal-isobaric)**
  - **Justification**: The NPT ensemble maintains constant pressure and temperature, which is crucial for simulating a solvated protein system under conditions that mimic experimental settings. This is particularly important for assessing the stability of the protein's folded structure in a realistic environment.
  - **Alternative Ensembles**: 
    - **NVT (canonical)**: Not chosen because it maintains constant volume, which does not account for pressure fluctuations that can occur in a solvated system.
    - **NVE (microcanonical)**: Not chosen because it does not control temperature or pressure, making it less suitable for simulating biological systems where these parameters are critical.

- **Thermostat: Langevin Thermostat**
  - **Justification**: The Langevin thermostat is effective at maintaining the temperature of the system and is particularly useful for simulating solvated systems. It also helps in damping out high-frequency vibrations

In [33]:
prompt_template_ms = """
DO NOT RUN ANYTHING — just answer.

If you were given the following prompt, choose the most appropriate
ensemble, thermostat, barostat, and force field. All will be simulated in OpenMM, so
choices should be compatible with OpenMM. 
IMPORTANT:
- If multiple stages are required, do the following for EACH stage. Otherwise, just do it once.
- Justify why your chosen ensemble is appropriate
- Explicitly state why alternative ensembles are NOT required
- Justify your choice of thermostat and barostat (if applicable) 
- Justify your choice of force field
- List any assumptions you are making about the system, and how those assumptions impact your choices.
- Avoid overly general statements; be specific to the system described in the prompt.


Respond with bullet points for each choice. Be concise, but specific.

Example prompt:
{prompt}
"""


prompt0_ms = """Simulate (in OpenMM) the thermophilic serine proteinase
(PDB: 1DBI), which is experimentally stable at 75 °C,
for 1 ns to assess whether its folded structure remains stable.
Plan a multi-stage simulation and justify each stage. 
"""

prompt0_ms = prompt_template_ms.format(prompt=prompt0_ms)
response0_ms = agent.run(prompt0_ms)
print("Response 0_ms:", response0_ms[0]["output"])

- **Stage 1: Energy Minimization**
  - **Ensemble:** Not applicable (energy minimization is not an ensemble-based process).
  - **Thermostat:** Not applicable.
  - **Barostat:** Not applicable.
  - **Force Field:** AMBER99SB-ILDN
    - **Justification:** This force field is well-suited for proteins and is compatible with OpenMM. It provides accurate representations of protein structures and dynamics.
  - **Assumptions:** The system is in a high-energy state initially and requires minimization to remove steric clashes and bad contacts.

- **Stage 2: Equilibration (NVT Ensemble)**
  - **Ensemble:** NVT (constant Number of particles, Volume, and Temperature)
    - **Justification:** This stage allows the system to reach thermal equilibrium at the desired temperature (75 °C) without changing the volume.
    - **Alternative Ensembles:** NPT is not required at this stage because we are focusing on temperature equilibration first.
  - **Thermostat:** Langevin Thermostat
    - **Justification: